# Thí nghiệm <model> / <method> / <expNNN>

Notebook này chạy một thí nghiệm rồi ghi kết quả. **Bấm Run all** là đủ.

Nó tự làm phần khó TRƯỚC khi chạy: kéo ĐÚNG bản code đã ghim ở cell đầu (nên chạy trên Colab hay trên máy cá nhân đều ra cùng kết quả), rồi kiểm dữ liệu, thiết bị và quyền ghi. Có gì chưa đúng thì nó **dừng và in ra danh sách việc phải sửa**, thay vì chạy nửa chừng rồi hỏng.

Cell đầu do `python scripts/pin.py` ghi - sửa tay sẽ bị ghi đè ở lần ghim sau. Các cell còn lại là bản mẫu trong `templates/experiment/notebook.ipynb`.

In [ ]:
# --- BẢN CODE ĐÃ GHIM (do scripts/pin.py ghi; sửa tay sẽ bị ghi đè) ---
REPO_URL = 'https://github.com/TrieuKhac-dev/SentimentX'
REPO_BRANCH = 'experiment'
REPO_SHA = 'd42de24d50934483fcbdd8da5c79c8b9c2935416'
EXP_DIR = 'qwen3-4b-instruct-2507/prompt-aspect/exp007'


In [ ]:
# Chuẩn bị môi trường: đưa thư mục code về ĐÚNG commit đã ghim rồi mới `import src`.
#
# PHẦN Ở LẠI Ô NÀY LÀ PHẦN KHÔNG THỂ ĐỂ Ở THƯ VIỆN: máy mới (Colab) chưa có `src/`, nên không thể gọi
# thư viện để kéo chính nó. Mọi việc chuẩn bị còn lại nằm trong `src/workflow/bootstrap.py`, và ô chỉ
# gọi 4 hàm theo đúng thứ tự cũ.
import importlib
import pathlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def repo_root():
    """Gốc repo: nơi có `configs/paths.yaml` và gói `src`.

    Nhận diện bằng CẤU HÌNH ĐƯỜNG DẪN chứ không bằng một file mã nguồn: `configs/paths.yaml` là nguồn
    duy nhất của mọi đường dẫn nên nó không chuyển nhà, còn file trong `src/` thì có thể. Trên Colab
    đây là thư mục code sẽ kéo về; trên máy cá nhân thì tìm từ thư mục đang đứng đi lên, để notebook
    chạy được dù Jupyter mở ở đâu.
    """
    if IN_COLAB:
        return pathlib.Path("/content/SentimentX")
    here = pathlib.Path.cwd()
    for candidate in [here] + list(here.parents):
        if (candidate / "configs" / "paths.yaml").is_file() and (candidate / "src" / "__init__.py").is_file():
            return candidate
    return here


def git(*args):
    """Chạy một lệnh git và in ra mã thoát (không giấu lỗi kéo code)."""
    done = subprocess.run(["git"] + [str(item) for item in args], capture_output=True, text=True)
    print("  git {} -> {}".format(" ".join(str(item) for item in args), done.returncode))
    if done.returncode != 0:
        print("    " + ((done.stdout or "") + (done.stderr or "")).strip()[-400:])
    return done.returncode


CODE_DIR = repo_root()


def has_source():
    """Đã có mã nguồn chưa - mốc nhận diện là `configs/paths.yaml` + gói `src`."""
    return (CODE_DIR / "configs" / "paths.yaml").is_file() and (CODE_DIR / "src" / "__init__.py").is_file()


def is_repo_here():
    """Thư mục code đã là git repo chưa (để khỏi clone đè lên repo có sẵn)."""
    done = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--is-inside-work-tree"],
                          capture_output=True, text=True)
    return done.returncode == 0 and done.stdout.strip() == "true"


if not has_source():
    print("Chưa có mã nguồn ở {} - đang kéo commit đã ghim...".format(CODE_DIR))
    # Chạy lại ô này là chuyện thường: Runtime -> Restart session, hoặc chạy lại sau khi sửa một lỗi
    # khác. Kéo code vào thư mục KHÔNG RỖNG mà không phải git repo thì git từ chối với mã thoát 128,
    # và dòng lỗi đỏ đó làm người đọc tưởng hỏng nặng - trong khi thư mục đó chỉ còn sót lại từ lần
    # chạy trước hỏng giữa chừng.
    if CODE_DIR.is_dir() and any(CODE_DIR.iterdir()) and not is_repo_here():
        raise SystemExit(
            "DỪNG: {} đã có sẵn nhưng KHÔNG phải git repo - nhiều khả năng là thư mục còn sót lại "
            "từ lần chạy trước hỏng giữa chừng.\nXoá nó rồi chạy lại ô này:\n  rm -rf {}".format(
                CODE_DIR, CODE_DIR))
    if is_repo_here():
        print("  thư mục đã là git repo - bỏ qua bước kéo mới, chỉ đi tới commit đã ghim")
    else:
        CODE_DIR.mkdir(parents=True, exist_ok=True)
        git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(CODE_DIR))
    git("-C", str(CODE_DIR), "fetch", "--depth", "1", "origin", REPO_SHA)
    git("-C", str(CODE_DIR), "checkout", "--detach", REPO_SHA)
if not has_source():
    # Không kéo được: in đủ thứ cần để đoán, thay vì chỉ báo thiếu file.
    print("KHÔNG thấy mã nguồn trong {}".format(CODE_DIR))
    print("  thư mục đang đứng : {}".format(pathlib.Path.cwd()))
    print("  CODE_DIR tồn tại  : {}".format(CODE_DIR.is_dir()))
    if CODE_DIR.is_dir():
        names = sorted(item.name for item in CODE_DIR.iterdir())
        print("  có trong đó       : {}".format(", ".join(names)[:400] or "(rỗng)"))
        root = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--show-toplevel"],
                              capture_output=True, text=True).stdout.strip()
        print("  là repo git       : {}".format(root or "không"))
    raise SystemExit(
        "DỪNG: không kéo được commit {} từ {}. Xem các dòng trên, hoặc Runtime -> Restart session "
        "rồi chạy lại.".format(REPO_SHA[:8], REPO_URL))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# XOÁ BỘ NHỚ ĐỆM IMPORT CŨ của kernel rồi mới import. Lần chạy ĐẦU TIÊN Python đã hỏi "ở thư mục này có
# gói nào không?" và ghi nhớ câu trả lời "không" vào `sys.path_importer_cache`; câu trả lời đó nằm
# trong bộ nhớ KERNEL nên `import src` VẪN báo `ModuleNotFoundError: No module named 'src'` dù file đã
# nằm trên đĩa, và chạy lại ô cũng không đổi. Đây đúng là lỗi đã gặp trên Colab.
importlib.invalidate_caches()
sys.path_importer_cache.pop(str(CODE_DIR), None)

# Việc chuẩn bị còn lại nằm trong THƯ VIỆN: 4 hàm, đúng thứ tự cũ (mount Drive và đặt gốc trước, rồi
# cài gói, rồi kiểm lại sha trước khi tải 27 MB tài nguyên model). Khối `with` bọc ngoài để ô này lỗi
# hay tự dừng thì cũng IN XONG rồi mới NGẮT phiên Colab - xem docs/00_workflow/10_template_notebook.md.
from src.api import bootstrap, end_session_on_error

with end_session_on_error():
    info = bootstrap.prepare()
    if info["stop"]:
        raise SystemExit(info["stop"])
    bootstrap.install_packages(EXP_DIR)
    bootstrap.verify_checkout(REPO_URL, REPO_SHA, REPO_BRANCH, CODE_DIR)
    bootstrap.model_assets(EXP_DIR)


In [ ]:
# Cấu hình ĐANG DÙNG: in ra để người đọc bảng điểm sau này biết nó thuộc cấu hình nào.

from src.api import dataset, end_session_on_error, experiments, model_config, paths, versioning

with end_session_on_error():
    # Cấu hình ĐANG DÙNG: in ra để người đọc bảng điểm sau này biết nó thuộc cấu hình nào.
    from src.api import dataset, experiments, model_config, paths, versioning

    parts = [part for part in EXP_DIR.replace("\\", "/").split("/") if part]
    model_id, method, exp_id = parts
    result = experiments.load(model_id, method, exp_id)
    config = result["config"]
    dataset_name = (config.get("data") or {}).get("dataset")
    if not dataset_name:
        raise SystemExit(
            "Thí nghiệm {!r} chưa khai `data.dataset`. Sửa config.yaml của thí nghiệm rồi chạy lại ô "
            "này.".format(EXP_DIR))
    ds = dataset.load_config(dataset_name)
    try:
        version_id = versioning.compute_id(ds)
    except Exception as exc:            # noqa: BLE001 - thiếu dữ liệu gốc là việc hay gặp trên Colab
        # In GỐC DỮ LIỆU ĐANG DÙNG rồi mới dừng. Đây là chi tiết quyết định: trên Colab gốc đó phải là thư
        # mục Drive, còn nếu phiên chưa mount Drive (hoặc chưa bấm Allow) thì nó trỏ vào máy ảo - và dữ
        # liệu gốc không có ở đó. Không in hai gốc ra thì người đọc chỉ thấy một traceback khó đoán.
        print("DỪNG: không tính được mã phiên bản dữ liệu.\n")
        print(exc)
        print("\nGốc dữ liệu đang dùng :", paths.data_root())
        print("Gốc kết quả đang dùng :", paths.results_root())
        print("Nếu hai gốc trên KHÔNG nằm trong thư mục Drive của bạn: phiên này chưa mount Drive, hoặc bạn")
        print("chưa bấm Allow khi Colab hỏi quyền. Restart session, bấm Run all, rồi bấm Allow.")
        raise SystemExit(1) from None
    approach = config.get("approach")

    # Đọc cấu hình bằng `.get`: model encoder KHÔNG có prompt, nên truy cập thẳng một khoá làm ô này
    # chết ngay, và chết TRƯỚC khi in ra bản code + gốc dữ liệu đang dùng - đúng lúc cần nhất.
    print("Thí nghiệm  : {}/{}/{}".format(model_id, method, exp_id))
    print("Đường chạy  : {}".format(approach))
    print("Dataset     : {} {} -> {}".format(ds["name"], ds.get("version"), version_id))
    print("Dữ liệu     : gốc {} | đã xử lý: {}".format(
        paths.data_root(), "có" if versioning.processed_dir(version_id).is_dir() else "CHƯA CÓ"))
    print("Vai         : {}".format((config.get("data") or {}).get("roles")))
    lora = config.get("lora") or {}
    if approach == "encoder":
        print("Huấn luyện  : trainer={}, LoRA r={} alpha={} trên {}, {} epoch, batch {} (tích luỹ {})".format(
            config.get("trainer"), lora.get("r"), lora.get("alpha"),
            ", ".join(lora.get("target_modules") or []), config.get("epochs"), config.get("batch"),
            config.get("grad_accum")))
        print("Prompt      : không có (model encoder học từ dữ liệu gán nhãn)")
    else:
        print("Prompt      : {}".format(config.get("prompt") or "(chưa khai)"))
        print("Ví dụ       : {}".format(config.get("examples") or "(không dùng ví dụ nào)"))
    print("Bài toán    : label_space={}, neutral_policy={}, not_mentioned={}".format(
        config.get("label_space"), config.get("neutral_policy"), config.get("not_mentioned")))
    print("Chấm điểm   : {}, chỉ số {}".format(
        "{} mẫu".format(config.get("n")) if config.get("n") else "cả split (n: null)",
        config.get("scores")))
    print("Ghi nhận    : tracker={}, experiment={}".format(
        config.get("tracker"), config.get("experiment")))
    print("Model       : ngưỡng cắt {} token".format(model_config.max_length(model_id)[0]))
    print("              nạp {}".format(model_config.inference(model_id)))
    print("\nLớp đã hợp nhất (theo thứ tự):")
    for label, path in result["layers"]:
        print("  {:<12} {}".format(label, path))
    if result["overrides"]:
        print("Khoá bị lớp sau đè lên:")
        for row in result["overrides"]:
            print("  ", row[0])


In [ ]:
# KIỂM TRƯỚC khi nạp model. Một lượt val tốn hàng chục phút, nên mọi thứ phải đúng từ đầu.
from src.api import end_session_on_error, preflight

with end_session_on_error():
    report = preflight.run(result, ds=ds, version_id=version_id, model_id=model_id,
                           method=method, exp_id=exp_id)
    preflight.print_report(report)
    print("\nThư mục kết quả:", report["info"].get("run_dir"))
    print("Trạng thái     :", report["info"].get("mode"), "-", report["info"].get("mode_reason"))
    if report["problems"]:
        # Khối `with` bọc ngoài NGẮT PHIÊN Colab sau khi mọi thứ đã in xong: ngắt phiên làm kernel mất
        # kết nối, nên ngắt trước khi in là giấu mất thứ người đọc cần nhất (sửa gì để chạy được).
        # Trên máy cá nhân hàm này không làm gì.
        raise SystemExit(
            "DỪNG: còn {} việc phải sửa (xem danh sách ở trên). Sửa xong thì chạy lại ô này.".format(
                len(report["problems"])))


In [ ]:
# --- Ô CHẠY THÍ NGHIỆM (scripts/run_notebook.py tìm theo dấu này) ---
import os

# CHẠY thí nghiệm rồi chấm điểm: gọi THƯ VIỆN của repo, không gọi script dòng lệnh và không chép lại
# logic. `src/experiments/experiment_run.py` là nơi DUY NHẤT biết cách chạy; cửa vào dòng lệnh (dùng
# khi muốn chạy nhanh ngoài notebook) cũng gọi đúng hai hàm dưới đây nên hai đường không thể lệch nhau.
from src.api import end_session_on_error, experiment_run, resume

# `SENTIMENTX_MODEL` (nếu đặt) cho phép máy này dùng bản trọng số có sẵn trên đĩa thay vì tải từ
# Hugging Face; bỏ trống thì dùng `checkpoint` trong config của model. Giá trị đã dùng nằm trong
# `run.log` và `run_meta.json` như mọi thông tin khác của lần chạy.
model_path = os.environ.get("SENTIMENTX_MODEL") or None

# 1) Chọn chế độ chạy - KHÔNG cần GPU (thiếu file hay sai config là dừng ngay). Ô này lỗi, hoặc tự
#    dừng, thì khối `with` ngắt phiên Colab sau khi đã in lý do.
with end_session_on_error():
    plan = experiment_run.plan(result, model=model_path)
    print("Chế độ chạy:", plan["mode"], "-", plan["reason"])
    if plan["mode"] == resume.MODE_STOP:
        raise SystemExit("DỪNG: " + plan["reason"])

# 2) Chạy thật. Lượt chạy lỗi thì THƯ VIỆN tự ngắt phiên (nhật ký đã đóng TRƯỚC khi ngắt), nên ở đây
#    KHÔNG bọc thêm - bọc nữa là ngắt hai lần cho cùng một lượt.
#    Máy đứt giữa chừng thì chạy lại ô này: phần đã xong nằm trong `predictions/part_*.jsonl` và điểm
#    số vẫn tính trên CẢ split.
run_result = experiment_run.run(plan)

# 3) In tóm tắt; vẫn bọc để lỗi ở bước in này cũng ngắt được phiên.
with end_session_on_error():
    print("\nChế độ:", run_result["mode"], "| thư mục:", run_result["out_dir"])


## Kết quả nằm ở đâu

Mỗi lần chạy một thư mục riêng: `experiments/<model>/<method>/<expNNN>/results/<hash8>/`.
`<hash8>` là 8 ký tự đầu của mã băm danh tính (cấu hình + dữ liệu + **commit đã ghim**), nên cùng một
phép đo trên Colab và trên máy cá nhân ra cùng tên thư mục.

| File | Nội dung |
| --- | --- |
| `run.log` | từng bước đã chạy, kèm lí do khi dừng; tìm `[RUN] mode=RESUME` khi chạy tiếp |
| `run_meta.json` | bản ghi lần chạy: code, config, dữ liệu, thiết bị, các attempt |
| `metrics.json` | chỉ số, kèm cách chấm (`label_space`, `neutral_policy`, số ô neutral bị loại) |
| `metrics.csv` | bảng dài `aspect, sentiment, metric, value` để so với các lần chạy khác |
| `mispredictions.csv` | chỉ các ô đoán sai |
| `predictions.csv` | từng review: prompt đã gửi model (đường prompt), nhãn đúng/đoán |
| `predictions/part_*.jsonl` | kết quả ghi dần; chạy lại thì đi tiếp từ đây |
| `model/last`, `model/best` | adapter LoRA (chỉ có ở thí nghiệm model encoder) |
| `errors.json` | CHỈ có khi lỗi: kiểu lỗi, vết gọi, thứ còn thiếu |

Trên DagsHub: mở địa chỉ **có đuôi `.mlflow`** (in ở ô cuối). Trang repo của DagsHub chỉ hiện dữ liệu
của DagsHub nên nhìn như rỗng - đó là hai trang khác nhau.


In [ ]:
# KẾT THÚC: nhắc lại chỗ cần xem, và việc cần làm trước khi giao.
import pathlib

from src.api import end_session, end_session_on_error, paths, tracking, utils

with end_session_on_error():
    # KẾT THÚC: nhắc lại chỗ cần xem, và việc cần làm trước khi giao.
    import pathlib

    from src.api import end_session, paths, tracking, utils

    print("Kết quả :", utils.rel(run_result["out_dir"]))
    # Chỉ in địa chỉ DagsHub khi lượt này THẬT SỰ được ghi lên đó. Bằng chứng nằm ở chính `run.log` (dòng
    # `[TRACK]`): thiếu thư viện `mlflow` hoặc thiếu token thì phần ghi nhận tự hạ cấp, và in địa chỉ trong
    # trường hợp đó là nói sai - đúng cảnh đã gặp ở ba lượt chạy đầu.
    _log = pathlib.Path(run_result["out_dir"]) / paths.pattern("run_log")
    _track = ([line for line in _log.read_text(encoding="utf-8", errors="replace").splitlines()
               if "[TRACK]" in line] if _log.is_file() else [])
    if _track and not all("không ghi nhận" in line for line in _track):
        print("DagsHub :", tracking.base.dagshub_config().get("mlflow_uri"))
    else:
        print("DagsHub : CHƯA ghi nhận lượt này - xem dòng [TRACK] trong run.log (thiếu `mlflow` hoặc token)")
    print("\nĐọc kết quả theo thứ tự: metrics.json (số chính) -> metrics.csv (so với lần chạy khác)")
    print("-> run.log (đã chạy những gì) -> errors.json (nếu có).")
    print("\nLưu ý: ghi lại file notebook này vào git (chỉ một file) rồi đẩy lên nhánh {}.\n"
          "Sau khi ghim thì KHÔNG sửa thư mục thí nghiệm nữa cho tới khi người nhận chạy xong."
          .format(REPO_BRANCH))

    # KẾT THÚC THẬT SỰ: ngắt phiên Colab để GPU thôi bị tính vào hạn mức, và in ra trạng thái để
    # người đọc biết vì sao phiên kết thúc. Trên máy cá nhân hàm này không làm gì.
    print(end_session())
